In [1]:
from tabulate import tabulate
import subprocess

In [2]:
def get_perf_stat(command, events):
    perf_l = ["perf", "stat", "-e", ",".join(events)]
    command_l = command.split()

    print(" ".join(perf_l + command_l))
    process = subprocess.Popen(perf_l + command_l,
                     stdout=subprocess.PIPE, 
                     stderr=subprocess.PIPE)
    #_, stderr = process.communicate()

    stdout, stderr = process.communicate()
    print(stdout.decode('UTF-8'))
    
    print(stderr)


    d = {}

    events_set = {e.upper() for e in events}

    print(events_set)
    
    for line in stderr.decode().split("\n"):
        l = line.strip().split()
        if len(l) < 2: continue
        if l[1].upper() in events_set:
            n_events = int(l[0].replace(",", ""))
            #percentage = 0 #float(l[-1][1:-3])
            d[l[1]] = n_events # , percentage)
    
    return d

In [3]:
def stats_normalize(d, n_steps):
    new_d = {}
    missing_events = []
    for e, p in d.items():
        try:
            new_d[e] = round(p/n_steps,1)
        except:
            missing_events.append(e)

    if len(missing_events) > 0:
        print("Not found these events {missing_events}")
    return new_d

def stats_compare(d_a, d_b, events):
    table = []
    for e in events:
        table.append([e, d_a[e], d_b[e]])
    print(tabulate(table))

In [4]:
def subtract_base(q, base):
    for k, v in base.items():
        q[k] = q[k]-v
        

To better understand memeory events take a look to [this](https://stackoverflow.com/questions/55035313/how-does-linux-perf-calculate-the-cache-references-and-cache-misses-events)

- LLC-load-misses and LLC-store-misses count only cacheable data read requests and RFO requests, respectively, that miss in the L3 cache.LLC-load-misses also includes reads for page walking. Both exclude hardware and software prefetching. (The difference compared to Haswell is that some types of prefetch requests are counted.)
-   'LONGEST_LAT_CACHE.REFERENCE',   # The number of loads on L3 cache, including hardware and software prefetching. (Equivalent to cache-references)
     'LONGEST_LAT_CACHE.MISS',        # The number of loads that miss L3 cache, including hardware and software prefetching. (Equivalent to cache-misses)
                                      # Both the previous include prefetch requests and code fetch requests that miss in the L3 cache. 
                                      # All of these events only count core-originating requests. They include requests from uops irrespective 
                                      # of whether end up retiring and irrespective of the source of the response. It's unclear to me how a
                                      # prefetch promoted to demand is counted

s
- Better descriptions of events and classified [here](https://www.cism.ucl.ac.be/Services/Formations/ICS/ics_2013.0.028/vtune_amplifier_xe/documentation/en/help/reference/index.htm#pmn/events/about_execution_performance_tuning_events.html)

In [23]:
events = [
     #'instructions', 
     'CPU_CLK_UNHALTED.THREAD',       # Counts the number of core cycles while the thread is not in a halt state. This event can approximate elapsed time while the core was not in the halt state.
     'CYCLE_ACTIVITY.STALLS_TOTAL',   # Total execution stalls
     'CYCLE_ACTIVITY.STALLS_L1D_MISS',# Execution stalls while L1 cache miss demand load is outstanding
     'CYCLE_ACTIVITY.STALLS_L2_MISS', # Execution stalls while L2 cache miss demand load is outstanding
     'CYCLE_ACTIVITY.STALLS_L3_MISS', # Execution stalls while L3 cache miss demand load is outstanding
     'CYCLE_ACTIVITY.STALLS_MEM_ANY', # Execution stalls while memory subsystem has an outstanding load
     #'CYCLE_ACTIVITY.CYCLES_MEM_ANY', # Cycles while memory subsystem has an outstanding load
     #'branches', 
     'branch-misses',
     'L1-dcache-loads',               # The number of loads in L3 cache 
     'L1-dcache-load-misses',         # The number of loads that miss L1 cache 
     'LLC-loads',                     # The number of loads in L3 cache (These are misses in both L1 and L2). Exclude hardware and software prefetching. 
     'LLC-load-misses',               # The number of loads that miss L3 cache. Exclude hardware and software prefetching.
     #'LONGEST_LAT_CACHE.REFERENCE',   # The number of loads on L3 cache, including hardware and software prefetching. (Equivalent to cache-references)
     #'LONGEST_LAT_CACHE.MISS',        # The number of loads that miss L3 cache, including hardware and software prefetching. (Equivalent to cache-misses)
                                      # Both the previous include prefetch requests and code fetch requests that miss in the L3 cache. 
                                      # All of these events only count core-originating requests. They include requests from uops irrespective 
                                      # of whether end up retiring and irrespective of the source of the response. It's unclear to me how a
                                      # prefetch promoted to demand is counted
     #'OFFCORE_REQUESTS_SQ_FULL',     # Offcore requests blocked due to Super Queue full. Counts the number of cycles that the SuperQ ( SQ ) is full. This event is thread agnostic. 
     #'MEM_INST_RETIRED.ALL_LOADS',
     #'LOAD_HIT_PRE.SW_PF',            # Counts load operations sent to the L1 data cache while a 
                                      # previous SSE prefetch instruction to the same cache line has 
                                      # started prefetching but has not yet finished.
                                      # LOAD_HIT_PRE.SW_PF (event 0x3C, umask 0x1) counts loads
                                      # fetched by the software prefetcher. So the counter lets you see how 
                                      # frequently your loads hit buffers fetched by software prefetch instructions.
     #'L2_LINES_OUT.USELESS_HWPF',
     #'L2_RQSTS.ALL_PF',
     #'L2_RQSTS.PF_HIT'               # L2_RQSTS.PF_HIT is incremented when an L2 hardware prefetch 
                                      # is issued and the corresponding line is 
                                      # found in the L2 cache. This could be considered a "wasted" 
                                      # prefetch, but since it is only accessing #
                                      # the tag array and not the data array it is unlikely to be a 
                                      # performance problem. I don't know if the # L2 hardware prefetchers 
                                      # modify their behavior when these hits occur.
     'L2_RQSTS.PF_MISS',             # L2_RQSTS.PF_MISS is incremented when an L2 hardware prefetch is
                                      # issued and the corresponding line is not found in the L2 cache.
                                      # This is the preferred case, and should be the most common.
                                      # An L2 prefetch that misses in the L2 is not guaranteed to bring the 
                                      # data into the L2 cache.  
                                      # As described in section 2.2.5 of the Intel Optimization Reference 
                                      # manual, L2 hardware prefetches 
                                      # always bring the data into the LLC, but do not always bring the 
                                      # data into the L2 cache.  
     #'L2_RQSTS.REFERENCES',
     'SW_PREFETCH_ACCESS.NTA',        # Counts the number of PREFETCHNTA instructions executed
     #'SW_PREFETCH_ACCESS.T0',        # Counts the number of PREFETCHT0 instructions executed
     #'SW_PREFETCH_ACCESS.T1_T2',     # Counts the number of PREFETCHT1 or PREFETCHT2 instructions executed
     #'SW_PREFETCH_ACCESS.PREFETCHW', # Counts the number of PREFETCHW instructions executed.
     #'RESOURCE_STALLS.ANY',          # https://stackoverflow.com/questions/60264057/what-resource-stall-other-might-mean
     #'RESOURCE_STALLS.SB'
]

In [24]:
n_runs = 10
n_queries = 3452

input_file = "/home/rossano/nq/data/documents.seismic_packed..ciao"
query_file = "/home/rossano/nq/data/queries.seismic"
query_cut = 1
heap_factor = 0.7

n_steps = n_runs * n_queries

In [25]:
command = f"../target/release/perf_inverted_index --index-file {input_file} --query-file {query_file} --query-cut {query_cut} --heap-factor {heap_factor} --n-queries {1}"

base = get_perf_stat(command, events) # Execute just one query

perf stat -e CPU_CLK_UNHALTED.THREAD,CYCLE_ACTIVITY.STALLS_TOTAL,CYCLE_ACTIVITY.STALLS_L1D_MISS,CYCLE_ACTIVITY.STALLS_L2_MISS,CYCLE_ACTIVITY.STALLS_L3_MISS,CYCLE_ACTIVITY.STALLS_MEM_ANY,branch-misses,L1-dcache-loads,L1-dcache-load-misses,LLC-loads,LLC-load-misses,L2_RQSTS.PF_MISS,SW_PREFETCH_ACCESS.NTA ../target/release/perf_inverted_index --index-file /home/rossano/nq/data/documents.seismic_packed..ciao --query-file /home/rossano/nq/data/queries.seismic --query-cut 1 --heap-factor 0.7 --n-queries 1
Searching for top-10 results
Number of evaluated queries: 1
Avg number of non-zero components: 50
Number of documents: 2680893
Avg number of non-zero components: 153
Time 144 microsecs per query

b"0\t1003688\t1\t9.575559\n0\t2285863\t2\t8.615733\n0\t2315299\t3\t8.005742\n0\t2285866\t4\t7.642974\n0\t1896604\t5\t7.63719\n0\t16671\t6\t7.4164815\n0\t2285868\t7\t7.0675445\n0\t2285865\t8\t7.054901\n0\t2285867\t9\t6.984206\n0\t2285871\t10\t6.8937793\n144\n\n Performance counter stats for '../targ

In [26]:
command = f"../target/release/perf_inverted_index --index-file {input_file} --query-file {query_file} --query-cut {query_cut} --heap-factor {heap_factor} --n-queries {n_queries} --n-runs {n_runs}"

d_query_p = get_perf_stat(command, events)

subtract_base(d_query_p, base)

perf stat -e CPU_CLK_UNHALTED.THREAD,CYCLE_ACTIVITY.STALLS_TOTAL,CYCLE_ACTIVITY.STALLS_L1D_MISS,CYCLE_ACTIVITY.STALLS_L2_MISS,CYCLE_ACTIVITY.STALLS_L3_MISS,CYCLE_ACTIVITY.STALLS_MEM_ANY,branch-misses,L1-dcache-loads,L1-dcache-load-misses,LLC-loads,LLC-load-misses,L2_RQSTS.PF_MISS,SW_PREFETCH_ACCESS.NTA ../target/release/perf_inverted_index --index-file /home/rossano/nq/data/documents.seismic_packed..ciao --query-file /home/rossano/nq/data/queries.seismic --query-cut 1 --heap-factor 0.7 --n-queries 3452 --n-runs 10


IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [27]:
stats_compare(d_query_p, d_query_p, events)

------------------------------  -----------  -----------
CPU_CLK_UNHALTED.THREAD         16405549458  16405549458
CYCLE_ACTIVITY.STALLS_TOTAL      3440924520   3440924520
CYCLE_ACTIVITY.STALLS_L1D_MISS   1445889990   1445889990
CYCLE_ACTIVITY.STALLS_L2_MISS    1261372055   1261372055
CYCLE_ACTIVITY.STALLS_L3_MISS     780864459    780864459
CYCLE_ACTIVITY.STALLS_MEM_ANY    2038555221   2038555221
branch-misses                      34487413     34487413
L1-dcache-loads                  8692777678   8692777678
L1-dcache-load-misses            1460432348   1460432348
LLC-loads                          53369192     53369192
LLC-load-misses                     5252222      5252222
L2_RQSTS.PF_MISS                  296494369    296494369
SW_PREFETCH_ACCESS.NTA              7718194      7718194
------------------------------  -----------  -----------


In [28]:
d_a = stats_normalize(d_query_p, n_steps)
d_b = stats_normalize(d_query_p, n_steps)

stats_compare(d_a, d_b, events)

------------------------------  --------  --------
CPU_CLK_UNHALTED.THREAD         475248    475248
CYCLE_ACTIVITY.STALLS_TOTAL      99679.2   99679.2
CYCLE_ACTIVITY.STALLS_L1D_MISS   41885.6   41885.6
CYCLE_ACTIVITY.STALLS_L2_MISS    36540.3   36540.3
CYCLE_ACTIVITY.STALLS_L3_MISS    22620.6   22620.6
CYCLE_ACTIVITY.STALLS_MEM_ANY    59054.3   59054.3
branch-misses                      999.1     999.1
L1-dcache-loads                 251819    251819
L1-dcache-load-misses            42306.8   42306.8
LLC-loads                         1546      1546
LLC-load-misses                    152.2     152.2
L2_RQSTS.PF_MISS                  8589.1    8589.1
SW_PREFETCH_ACCESS.NTA             223.6     223.6
------------------------------  --------  --------


In [57]:
command = f"../target/release/perf --n {n} --update"

d_update = get_perf_stat(command, events)

perf stat -e CPU_CLK_UNHALTED.THREAD,CYCLE_ACTIVITY.STALLS_MEM_ANY,CYCLE_ACTIVITY.CYCLES_MEM_ANY,L1-dcache-loads,L1-dcache-load-misses,LLC-loads,LLC-load-misses,LONGEST_LAT_CACHE.REFERENCE,LONGEST_LAT_CACHE.MISS,LOAD_HIT_PRE.SW_PF ../target/release/perf --n 1000000000 --update
b"\n Performance counter stats for '../target/release/perf --n 1000000000 --update':\n\n    43,638,285,944      CPU_CLK_UNHALTED.THREAD                                                 (49.93%)\n    30,873,694,418      CYCLE_ACTIVITY.STALLS_MEM_ANY                                           (49.94%)\n    40,097,953,649      CYCLE_ACTIVITY.CYCLES_MEM_ANY                                           (49.94%)\n     3,566,687,704      L1-dcache-loads                                                         (49.98%)\n     1,251,314,777      L1-dcache-load-misses            #   35.08% of all L1-dcache accesses   (50.02%)\n       816,552,017      LLC-loads                                                               (40.05%)

In [58]:
stats_compare(d_sum, d_update, events)

-----------------------------  -----------  -----------
CPU_CLK_UNHALTED.THREAD        17063786461  43638285944
CYCLE_ACTIVITY.STALLS_MEM_ANY   9086166496  30873694418
CYCLE_ACTIVITY.CYCLES_MEM_ANY  14849916525  40097953649
L1-dcache-loads                 1735581114   3566687704
L1-dcache-load-misses            355992843   1251314777
LLC-loads                        264470320    816552017
LLC-load-misses                    7216147    380568248
LONGEST_LAT_CACHE.REFERENCE     1028701970   2595495119
LONGEST_LAT_CACHE.MISS            47155401   1687067122
LOAD_HIT_PRE.SW_PF                   12679        40997
-----------------------------  -----------  -----------


In [59]:
d_a = stats_normalize(d_sum, n_steps)
d_b = stats_normalize(d_update, n_steps)

stats_compare(d_a, d_b, events)

-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        341.3  872.8
CYCLE_ACTIVITY.STALLS_MEM_ANY  181.7  617.5
CYCLE_ACTIVITY.CYCLES_MEM_ANY  297    802
L1-dcache-loads                 34.7   71.3
L1-dcache-load-misses            7.1   25
LLC-loads                        5.3   16.3
LLC-load-misses                  0.1    7.6
LONGEST_LAT_CACHE.REFERENCE     20.6   51.9
LONGEST_LAT_CACHE.MISS           0.9   33.7
LOAD_HIT_PRE.SW_PF               0      0
-----------------------------  -----  -----


-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        435.3  968.4
CYCLE_ACTIVITY.STALLS_MEM_ANY  283.4  724.7
CYCLE_ACTIVITY.CYCLES_MEM_ANY  391    897.1
L1-dcache-loads                 36     71.8
L1-dcache-load-misses            9.5   34.8
LLC-loads                        6.9   21.5
LLC-load-misses                  0.4   11.3
LONGEST_LAT_CACHE.REFERENCE     26.2   56.7
LONGEST_LAT_CACHE.MISS           1.8   34.6
LOAD_HIT_PRE.SW_PF               0      0
-----------------------------  -----  -----

English.100Mb 4 Mb extra
English.50Mb 2 Mb extra info
English.5MB 200K

English.32Mb
-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        390.7  241.8
CYCLE_ACTIVITY.STALLS_MEM_ANY  264.3   87.7
CYCLE_ACTIVITY.CYCLES_MEM_ANY  383.3  226.6
L1-dcache-loads                 38.1   53
L1-dcache-load-misses            5      5.5
LLC-loads                        2.7    1.5
LLC-load-misses                  0.8    0.1
LONGEST_LAT_CACHE.REFERENCE     12.8   11.9
LONGEST_LAT_CACHE.MISS           4.8    4.1
LOAD_HIT_PRE.SW_PF               0      1.4
SW_PREFETCH_ACCESS.NTA           0      3.9
SW_PREFETCH_ACCESS.T0            0      0
-----------------------------  -----  -----
Time:                           294    177   66%


English.50Mb

Time:                           344    203   69%

English.100Mb
-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        508.7  390.6
CYCLE_ACTIVITY.STALLS_MEM_ANY  359.1  212.4
CYCLE_ACTIVITY.CYCLES_MEM_ANY  499.1  366.6
L1-dcache-loads                 42.7   55.2
L1-dcache-load-misses            6      6.7
LLC-loads                        3.3    2.1
LLC-load-misses                  1.6    0.5 <- Almost no L3 cache misses
LONGEST_LAT_CACHE.REFERENCE     15.9   15.1
LONGEST_LAT_CACHE.MISS           9.3    8.1
LOAD_HIT_PRE.SW_PF               0      0.9
SW_PREFETCH_ACCESS.NTA           0      4.1
SW_PREFETCH_ACCESS.T0            0      0
-----------------------------  -----  -----
Time:                           391    286   36%

English.100Mb Accessing only superblocks
-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        515.6  155.4
CYCLE_ACTIVITY.STALLS_MEM_ANY  372.3   89.9
CYCLE_ACTIVITY.CYCLES_MEM_ANY  507.1  152.2
LLC-loads                        3.3    1.5
LLC-load-misses                  1.6    0.2 <- No L3-misses. This means that Superblocks are in cache
LONGEST_LAT_CACHE.REFERENCE     15.8    9.7 
LONGEST_LAT_CACHE.MISS           9.3    4.2 <- Includes HW ad SW prefetch
LOAD_HIT_PRE.SW_PF               0      0
SW_PREFETCH_ACCESS.NTA           0      3.4 
SW_PREFETCH_ACCESS.T0            0      0
-----------------------------  -----  -----

English.1Mb
-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        140.2  152.4
CYCLE_ACTIVITY.STALLS_MEM_ANY   32.7   15.2
CYCLE_ACTIVITY.CYCLES_MEM_ANY  130.1  130.5
L1-dcache-loads                 37.5   49.9
L1-dcache-load-misses            2.8    4.1
LLC-loads                        1.4    0.3
LLC-load-misses                  0      0
LONGEST_LAT_CACHE.REFERENCE      5.8    6.3
LONGEST_LAT_CACHE.MISS           0.1    0.1
LOAD_HIT_PRE.SW_PF               0      0
SW_PREFETCH_ACCESS.NTA           0      4
SW_PREFETCH_ACCESS.T0            0      0
-----------------------------  -----  -----


English.5Mb
-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        144.7  166.1
CYCLE_ACTIVITY.STALLS_MEM_ANY   37.1   24.3
CYCLE_ACTIVITY.CYCLES_MEM_ANY  136.5  146.7
L1-dcache-loads                 37.8   50.4
L1-dcache-load-misses            3.2    4.1
LLC-loads                        2.1    0.9 <- Prefetching is working to prefetch data from L3 to lower levels but it's not enough
LLC-load-misses                  0      0   <-- no L3 cache miss, everything fits in L3 (16 MiB)
LONGEST_LAT_CACHE.REFERENCE      8.8    8.7
LONGEST_LAT_CACHE.MISS           0.1    0.1
LOAD_HIT_PRE.SW_PF               0      0
SW_PREFETCH_ACCESS.NTA           0      4.1
SW_PREFETCH_ACCESS.T0            0      0
-----------------------------  -----  -----
Time                           111     130


English.10Mb
-----------------------------  -----  -----
CPU_CLK_UNHALTED.THREAD        156.7  179.7
CYCLE_ACTIVITY.STALLS_MEM_ANY   49.7   37.9
CYCLE_ACTIVITY.CYCLES_MEM_ANY  150.1  161.3
L1-dcache-loads                 36.5   49.4 
L1-dcache-load-misses            3.9    4.7 
LLC-loads                        2.2    1.1  
LLC-load-misses                  0      0   <-- no L3 cache miss, everything fits in L3 (16 MiB)
LONGEST_LAT_CACHE.REFERENCE     10.2    9.8
LONGEST_LAT_CACHE.MISS           0.2    0.5
LOAD_HIT_PRE.SW_PF               0      0.2
SW_PREFETCH_ACCESS.NTA           0      4.1
SW_PREFETCH_ACCESS.T0            0      0
-----------------------------  -----  -----
Time                           123     135



In [ ]:
Possible improvments:
    - There is no need to compute superlock occs again
    - Add extra data to prefetch superblocks


In [ ]:
~/pmu-tools (master) » python3 toplev.py --core S0-C0 -l1 -v --no-desc taskset -c 0 ../qwt_private/target/release/perf_wavelet_tree --input-file /data1/Texts/pizzachili/english/english.100MB --rank-prefetch --n-queries 10000000  
Consider disabling nmi watchdog to minimize multiplexing
(echo 0 | sudo tee /proc/sys/kernel/nmi_watchdog or
 echo kernel.nmi_watchdog=0 >> /etc/sysctl.conf ; sysctl -p as root)
Will measure complete system.
Text length: 104857600
Alphabet size: 215
Wavelet tree already exists. Filename: /data1/Texts/pizzachili/english/english.100MB.256.qwt. I'm going to read it ...
Serialized size: 118170353 bytes
t_min: 283 fake result: 95572
# 4.5-full-perf on Intel(R) Core(TM) i9-9900KF CPU @ 3.60GHz [cfl/skylake]
C0    FE             Frontend_Bound   % Slots                       6.5  < [33.3%]
C0    BAD            Bad_Speculation  % Slots                       7.7  < [33.3%]
C0    BE             Backend_Bound    % Slots                      60.4    [33.3%]<==
C0    RET            Retiring         % Slots                      25.3  < [33.3%]
C0-T0 MUX                             %                            33.33  
C0-T1 MUX                             %                            33.33  
Run toplev --describe Backend_Bound^ to get more information on bottleneck


Serialized size: 118170353 bytes
t_min: 283 fake result: 280641
# 4.5-full-perf on Intel(R) Core(TM) i9-9900KF CPU @ 3.60GHz [cfl/skylake]
C0    FE             Frontend_Bound                      % Slots                       6.6  < [ 7.7%]
C0    BAD            Bad_Speculation                     % Slots                       7.8  < [ 7.7%]
C0    BE             Backend_Bound                       % Slots                      60.3    [ 7.7%]
C0    RET            Retiring                            % Slots                      25.5  < [ 7.7%]
C0    FE             Frontend_Bound.Fetch_Latency        % Slots                       3.1  < [ 7.7%]
C0    FE             Frontend_Bound.Fetch_Bandwidth      % Slots                       3.5  < [ 7.7%]
C0    BAD            Bad_Speculation.Branch_Mispredicts  % Slots                       7.8  < [ 7.7%]
C0    BAD            Bad_Speculation.Machine_Clears      % Slots                       0.0  < [ 7.7%]
C0    BE/Mem         Backend_Bound.Memory_Bound          % Slots                      43.9    [ 7.7%]<==
C0    BE/Core        Backend_Bound.Core_Bound            % Slots                      16.4    [ 7.7%]
C0    RET            Retiring.Light_Operations           % Slots                      23.9  < [ 7.7%]
C0    RET            Retiring.Heavy_Operations           % Slots                       1.5  < [ 7.7%]
C0-T0 MUX                                                %                             7.69  
C0-T1 MUX                                                %                             7.69 


Wavelet tree already exists. Filename: /data1/Texts/pizzachili/english/english.100MB.256.qwt. I'm going to read it ...
Serialized size: 118170353 bytes
t_min: 282 fake result: 2375594
# 4.5-full-perf on Intel(R) Core(TM) i9-9900KF CPU @ 3.60GHz [cfl/skylake]
C0    FE             Frontend_Bound                                   % Slots                       6.6  < [ 2.6%]
C0    BAD            Bad_Speculation                                  % Slots                       7.9  < [ 2.6%]
C0    BE             Backend_Bound                                    % Slots                      60.3    [ 2.6%]
C0    RET            Retiring                                         % Slots                      25.5  < [ 2.6%]
C0    FE             Frontend_Bound.Fetch_Latency                     % Slots                       3.1  < [ 2.6%]
C0    FE             Frontend_Bound.Fetch_Bandwidth                   % Slots                       3.5  < [ 2.6%]
C0    BAD            Bad_Speculation.Branch_Mispredicts               % Slots                       7.8  < [ 2.6%]
C0    BAD            Bad_Speculation.Machine_Clears                   % Slots                       0.0  < [ 2.6%]
C0    BE/Mem         Backend_Bound.Memory_Bound                       % Slots                      43.9    [ 2.6%]
C0    BE/Core        Backend_Bound.Core_Bound                         % Slots                      16.3    [ 2.6%]
C0    RET            Retiring.Light_Operations                        % Slots                      24.2  < [ 2.6%]
C0    RET            Retiring.Heavy_Operations                        % Slots                       1.5  < [ 2.6%]
C0    RET            Retiring.Light_Operations.Memory_Operations      % Slots                       3.5  < [ 2.6%]
C0    RET            Retiring.Light_Operations.Fused_Instructions     % Slots                       1.5  < [ 2.6%]
C0    RET            Retiring.Light_Operations.Non_Fused_Branches     % Slots                       0.6  < [ 2.6%]
C0    RET            Retiring.Light_Operations.Nop_Instructions       % Slots                       0.1  < [ 2.6%]
C0    RET            Retiring.Light_Operations.Other_Light_Ops        % Slots                      18.4  < [ 2.6%]
C0    RET            Retiring.Heavy_Operations.Few_Uops_Instructions  % Slots                       1.3  < [ 2.6%]
C0    RET            Retiring.Heavy_Operations.Microcode_Sequencer    % Slots                       0.2  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Latency.ICache_Misses       % Clocks                      0.1  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Latency.ITLB_Misses         % Clocks                      0.0  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Latency.Branch_Resteers     % Clocks_est                  2.8  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Latency.DSB_Switches        % Clocks                      4.0  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Latency.LCP                 % Clocks                      0.0  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Latency.MS_Switches         % Clocks                      0.3  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Bandwidth.MITE              % Slots_est                   7.8  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Bandwidth.DSB               % Slots_est                   3.9  < [ 2.6%]
C0-T0 FE             Frontend_Bound.Fetch_Bandwidth.LSD               % Slots_est                   0.0  < [ 2.6%]
C0-T0 BE/Mem         Backend_Bound.Memory_Bound.L1_Bound              % Stalls                      3.5  < [ 2.6%]
C0-T0 BE/Mem         Backend_Bound.Memory_Bound.L2_Bound              % Stalls                      6.2    [ 2.6%]
C0-T0 BE/Mem         Backend_Bound.Memory_Bound.L3_Bound              % Stalls                      8.8    [ 2.6%]
C0-T0 BE/Mem         Backend_Bound.Memory_Bound.DRAM_Bound            % Stalls                     34.9    [ 2.6%]<==
C0-T0 BE/Mem         Backend_Bound.Memory_Bound.Store_Bound           % Stalls                      0.2  < [ 2.6%]
C0-T0 BE/Core        Backend_Bound.Core_Bound.Divider                 % Clocks                      0.5  < [ 2.6%]
C0-T0 BE/Core        Backend_Bound.Core_Bound.Ports_Utilization       % Clocks                     20.2    [ 2.6%]
C0-T0 RET            Retiring.Light_Operations.FP_Arith               % Uops                        0.0  < [ 2.6%]
C0-T0 MUX                                                             %                             2.63  
C0-T1 FE             Frontend_Bound.Fetch_Latency.ICache_Misses       % Clocks                     10.1  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Latency.ITLB_Misses         % Clocks                      4.7  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Latency.Branch_Resteers     % Clocks_est                  9.0  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Latency.DSB_Switches        % Clocks                      0.3  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Latency.LCP                 % Clocks                      0.0  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Latency.MS_Switches         % Clocks                      7.3  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Bandwidth.MITE              % Slots_est                   0.0  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Bandwidth.DSB               % Slots_est                   0.0  < [ 2.6%]
C0-T1 FE             Frontend_Bound.Fetch_Bandwidth.LSD               % Slots_est                   0.0  < [ 2.6%]
C0-T1 BE/Mem         Backend_Bound.Memory_Bound.L1_Bound              % Stalls                      7.7  < [ 2.6%]
C0-T1 BE/Mem         Backend_Bound.Memory_Bound.L2_Bound              % Stalls                      0.3  < [ 2.6%]
C0-T1 BE/Mem         Backend_Bound.Memory_Bound.L3_Bound              % Stalls                      8.7    [ 2.6%]
C0-T1 BE/Mem         Backend_Bound.Memory_Bound.DRAM_Bound            % Stalls                     11.9    [ 2.6%]
C0-T1 BE/Mem         Backend_Bound.Memory_Bound.Store_Bound           % Stalls                      1.5  < [ 2.6%]
C0-T1 BE/Core        Backend_Bound.Core_Bound.Divider                 % Clocks                      0.3  < [ 2.6%]
C0-T1 BE/Core        Backend_Bound.Core_Bound.Ports_Utilization       % Clocks                     19.1    [ 2.6%]
C0-T1 RET            Retiring.Light_Operations.FP_Arith               % Uops                        0.0  < [ 2.6%]
C0-T1 MUX                                                             %                             2.63  
warning: 1 results not referenced: 45
Run toplev --describe DRAM_Bound^ Memory_Bound^ to get more information on bottlenecks
Add --run-sample to find locations
Add --nodes '!+DRAM_Bound*/4,+Memory_Bound*/3,+Backend_Bound.Memory_Bound,+Backend_Bound,+Backend_Bound,+MUX' for breakdown.




